# Smart Course Navigator - Model Training

This notebook executes the training pipeline and records evaluation metrics for the final report screenshots.

Run cells from the **project root** (`smart-course-navigator/`) so `utils` resolves, or prepend the root to `sys.path` as shown below.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent  # if launched from notebooks/
sys.path.insert(0, str(ROOT))

from utils.paths import DATASET_CSV, METRICS_PATH, MODEL_PATH
from utils.ml_pipeline import MLPipeline

print("Dataset:", DATASET_CSV.exists(), DATASET_CSV)

Dataset: True /home/txdigitalafrica/Downloads/BI semester project/smart-course-navigator/data/dataset.csv


In [2]:
pipeline = MLPipeline(random_state=42)
result = pipeline.train_and_save(DATASET_CSV, MODEL_PATH, METRICS_PATH)
result.metrics

{'selected_model': 'random_forest',
 'selection_criterion': 'Highest F1-score on stratified hold-out test set (imbalance handled via class_weight).',
 'results': [{'model': 'logistic_regression',
   'accuracy': 0.6507,
   'precision': 0.851,
   'recall': 0.639,
   'f1_score': 0.7299},
  {'model': 'decision_tree',
   'accuracy': 0.6027,
   'precision': 0.7883,
   'recall': 0.6318,
   'f1_score': 0.7014},
  {'model': 'random_forest',
   'accuracy': 0.712,
   'precision': 0.8189,
   'recall': 0.7834,
   'f1_score': 0.8007}],
 'feature_columns': ['gpa',
  'course_difficulty',
  'course_level',
  'attempts',
  'attendance',
  'credit_load',
  'program',
  'course_id'],
 'feature_importance': [{'feature': 'num__gpa',
   'importance': 0.4627280429984315},
  {'feature': 'num__attendance', 'importance': 0.16031329176228137},
  {'feature': 'num__course_difficulty', 'importance': 0.15830558121357466},
  {'feature': 'num__course_level', 'importance': 0.042816694512720015},
  {'feature': 'num__atte

In [3]:
import pandas as pd
from pprint import pprint

print("Selected model:")
pprint(result.metrics["selected_model"])
print("\nSelection criterion:")
pprint(result.metrics["selection_criterion"])

results_df = pd.DataFrame(result.metrics["results"])
results_df

Selected model:
'random_forest'

Selection criterion:
('Highest F1-score on stratified hold-out test set (imbalance handled via '
 'class_weight).')


,model,accuracy,precision,recall,f1_score
0,logistic_regression,0.6507,0.8510,0.6390,0.7299
1,decision_tree,0.6027,0.7883,0.6318,0.7014
2,random_forest,0.7120,0.8189,0.7834,0.8007


In [4]:
feature_df = pd.DataFrame(result.metrics["feature_importance"])
feature_df.head(12)

,feature,importance
0,num__gpa,0.462728
1,num__attendance,0.160313
2,num__course_difficulty,0.158306
3,num__course_level,0.042817
4,num__attempts,0.036641
5,cat__course_id_GL201,0.010598
6,cat__course_id_ME401,0.010476
7,cat__program_Mining Engineering,0.010200
8,cat__course_id_ME301,0.010186
9,cat__program_Electrical Engineering,0.009650


In [5]:
from pathlib import Path
import matplotlib.pyplot as plt

out_dir = Path("screenshots/notebook")
out_dir.mkdir(parents=True, exist_ok=True)

# Save model comparison table as PNG
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.axis("off")
ax.table(
    cellText=results_df.round(4).values,
    colLabels=results_df.columns,
    loc="center",
)
ax.set_title("Model Comparison Metrics", fontsize=12, pad=12)
fig.tight_layout()
fig.savefig(out_dir / "01_model_metrics_table.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Save feature importance table as PNG
feat_show = feature_df.head(12).copy().round(6)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.axis("off")
ax.table(
    cellText=feat_show.values,
    colLabels=feat_show.columns,
    loc="center",
)
ax.set_title("Top Feature Importances", fontsize=12, pad=12)
fig.tight_layout()
fig.savefig(out_dir / "02_feature_importance_table.png", dpi=220, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_dir / "01_model_metrics_table.png")
print("Saved:", out_dir / "02_feature_importance_table.png")

Saved: screenshots/notebook/01_model_metrics_table.png
Saved: screenshots/notebook/02_feature_importance_table.png
